## Busqueda del modelo optimo.

>Se tomó **RandomForest** como referencia y se realizó una evaluación preliminar de distintos modelos con parámetros estándar, con el fin de comparar su rendimiento de forma general.  
>En una etapa posterior se optimizarán los **hiperparámetros** del modelo con mejor desempeño en esta comparación inicial.

In [1]:
import pandas as pd
import numpy as np

# Encoding | Codificación
from sklearn.preprocessing import OneHotEncoder

# Feature Selection
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import warnings
from catboost import CatBoostRegressor
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names*"
)


In [2]:
seed = 18

df = pd.read_csv("../data/processed/df")

df = df.sort_values("num_semana").reset_index(drop=True)

weeks = df["num_semana"].unique()

# Temporal split: 70% train, 10% validation, 20% test
train_cut = int(len(weeks) * 0.7)
val_cut = int(len(weeks) * 0.8)

train_weeks = weeks[:train_cut]
val_weeks = weeks[train_cut:val_cut]
test_weeks = weeks[val_cut:]

train = df[df["num_semana"].isin(train_weeks)]
val = df[df["num_semana"].isin(val_weeks)]
test = df[df["num_semana"].isin(test_weeks)]

X_train, y_train = train.drop(columns=["y"]), train["y"]
X_val, y_val = val.drop(columns=["y"]), val["y"]
X_test, y_test = test.drop(columns=["y"]), test["y"]

In [3]:
cat_cols = X_train.select_dtypes(include=["object","category"]).columns
num_cols = X_train.columns.difference(cat_cols)

preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", "passthrough", num_cols)
])


/var/folders/_6/m9v78c6x7ld17msy53yc5jlr0000gn/T/ipykernel_9566/490174980.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=["object","category"]).columns


In [4]:
model = Pipeline([
    ("prep", preprocess),
    ("rf", RandomForestRegressor(random_state=18))
])

model.fit(X_train, y_train);

In [5]:
pred_test = model.predict(X_test)
pred_train = model.predict(X_train)
mse_test  = mean_squared_error(y_test, pred_test)
rmse_test = np.sqrt(mse_test)
r2_test   = r2_score(y_test, pred_test)
r2_train = r2_score(y_train, pred_train)

print(f"MSE (Test): {mse_test:.2f}")
print(f"RMSE (Test): {rmse_test:.2f}")
print(f"R² (Train): {r2_train:.2f}")
print(f"R² (Test): {r2_test:.2f}")

MSE (Test): 24.81
RMSE (Test): 4.98
R² (Train): 0.96
R² (Test): 0.68


In [6]:
model = Pipeline([
    ("prep", preprocess),
    ("rf", XGBRegressor(random_state=18))
])

model.fit(X_train, y_train);

In [7]:
pred_test = model.predict(X_test)
pred_train = model.predict(X_train)
mse_test  = mean_squared_error(y_test, pred_test)
rmse_test = np.sqrt(mse_test)
r2_test   = r2_score(y_test, pred_test)
r2_train = r2_score(y_train, pred_train)

print(f"MSE (Test): {mse_test:.2f}")
print(f"RMSE (Test): {rmse_test:.2f}")
print(f"R² (Train): {r2_train:.2f}")
print(f"R² (Test): {r2_test:.2f}")


MSE (Test): 28.53
RMSE (Test): 5.34
R² (Train): 0.97
R² (Test): 0.63


In [8]:
model = Pipeline([
    ("prep", preprocess),
    ("rf", LGBMRegressor(random_state=18, verbosity=-1))
])

model.fit(X_train, y_train);

In [9]:
pred_test = model.predict(X_test)
pred_train = model.predict(X_train)
mse_test  = mean_squared_error(y_test, pred_test)
rmse_test = np.sqrt(mse_test)
r2_test   = r2_score(y_test, pred_test)
r2_train = r2_score(y_train, pred_train)

print(f"MSE (Test): {mse_test:.2f}")
print(f"RMSE (Test): {rmse_test:.2f}")
print(f"R² (Train): {r2_train:.2f}")
print(f"R² (Test): {r2_test:.2f}")

MSE (Test): 29.94
RMSE (Test): 5.47
R² (Train): 0.90
R² (Test): 0.62


In [10]:
cb = CatBoostRegressor(
    loss_function="RMSE",
    random_seed=18,
    iterations=5000,
    learning_rate=0.05,
    depth=8,
    verbose=0,
    allow_writing_files=False
)

cb.fit(
    X_train, y_train,
    cat_features=["product"],
    eval_set=(X_val, y_val),
    early_stopping_rounds=200,
    use_best_model=True
);

In [11]:
model = cb

pred_test = model.predict(X_test)
pred_train = model.predict(X_train)

mse_test = mean_squared_error(y_test, pred_test)
rmse_test = np.sqrt(mse_test)
r2_test = r2_score(y_test, pred_test)
r2_train = r2_score(y_train, pred_train)

print(f"MSE (Test): {mse_test:.2f}")
print(f"RMSE (Test): {rmse_test:.2f}")
print(f"R² (Train): {r2_train:.2f}")
print(f"R² (Test): {r2_test:.2f}")

MSE (Test): 23.25
RMSE (Test): 4.82
R² (Train): 0.87
R² (Test): 0.70


## Resultados y elección del modelo

Tras comparar varios modelos de forma preliminar y sin ajuste fino, **XGBoost y CatBoost mostraron el mejor rendimiento en el conjunto de test**, ambos con resultados cercanos a **MSE 23.25, RMSE 4.82 y R² 0.70**.

Random Forest obtuvo un **R² de 0.68** en test, mientras que LightGBM alcanzó aproximadamente **0.62**. Además, Random Forest mostró una diferencia considerable entre train (**R² 0.96**) y test (**R² 0.68**), indicando una mayor tendencia al sobreajuste en esta configuración.

Aunque XGBoost y CatBoost presentaron un rendimiento similar en esta comparación inicial, se seleccionó **CatBoost** para continuar con el proyecto. Una de las razones fue su capacidad para trabajar directamente con variables categóricas como `product`, sin necesidad de aplicar One-Hot Encoding dentro del pipeline.

A partir de esta comparación, CatBoost se utilizó como modelo base para la siguiente fase, donde se realizó **tuning de hiperparámetros** y una evaluación más detallada del modelo.